<a href="https://colab.research.google.com/github/douglasmazzinghy/2026-SBPO/blob/main/mclaughlin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Sequential Algorithm for Open-Pit Mine Scheduling Problem

### Installing packages

In [ ]:
using Pkg
Pkg.add([
        PackageSpec(name="JuMP", version="1.26.0"),         # Optimization
        PackageSpec(name="Gurobi", version="1.7.4"),        # Solver
        PackageSpec(name="CSV", version="0.10.15"),         # Files
        PackageSpec(name="DataFrames", version="1.7.0"),    # Tables
        PackageSpec(name="Format", version="1.3.7"),        # Results Format
        PackageSpec(name="PrettyTables", version="2.4.0"),  # Tables
        PackageSpec(name="Statistics", version="1.11.1"),   # Statistics
        PackageSpec(name="PlotlyJS", version="0.18.16")     # Plots
        ])

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


### Importing packages

In [ ]:
using JuMP, Gurobi, CSV, DataFrames, Format, PrettyTables, Statistics, PlotlyJS

WebIO._IJuliaInit()

### Loading Gurobi Solver License (file gurobi.lic)

In [ ]:
# Upload the Gurobi licence file "gurobi.lic"
# In the left side menu, click the folder icon.
# Use the upload icon (usually an up arrow) or drag the gurobi.lic file from your local machine to the side menu area.
ENV["GRB_LICENSE_FILE"] = "/content/gurobi.lic"

"/content/gurobi.lic"

### Loading block model file

In [ ]:
download("https://mansci-web.uai.cl/minelib/data/mclaughlin.blocks", "mclaughlin.blocks")

"mclaughlin.blocks"

In [ ]:
# Load block model
blocks = CSV.read("mclaughlin.blocks", DataFrame, delim=' ',
              header=[:id, :x, :y, :z, :blockvalue, :tonn, :destination, :Au],
              types=Dict(:id => Int, :x => Float64, :y => Float64, :z => Float64,
                        :blockvalue => Float64, :tonn => Float64, :destination => Int,
                        :Au => Float64))
blocks.upit = zeros(Int, nrow(blocks))
println("Loaded: $(nrow(blocks)) blocks")
blocks

Loaded: 2140342 blocks


Row,id,x,y,z,blockvalue,tonn,destination,Au,upit
,Int64,Float64,Float64,Float64,Float64,Float64,Int64,Float64,Int64
1,0,36.0,286.0,67.0,-27.0,20.83,0,0.0,0
2,1,37.0,286.0,67.0,-117.0,83.33,0,0.0,0
3,2,38.0,286.0,67.0,-48.0,31.25,0,0.0,0
4,3,33.0,287.0,67.0,-7.0,10.42,0,0.0,0
5,4,34.0,287.0,67.0,-96.0,72.92,0,0.0,0
6,5,35.0,287.0,67.0,-186.0,135.42,0,0.0,0
7,6,36.0,287.0,67.0,-275.0,208.33,0,0.0,0
8,7,37.0,287.0,67.0,-357.0,270.83,0,0.0,0
9,8,38.0,287.0,67.0,-213.0,166.67,0,0.0,0


### Loading block precedences file

In [ ]:
download("https://mansci-web.uai.cl/minelib/data/mclaughlin.prec.zip", "mclaughlin.prec.zip")

"mclaughlin.prec.zip"

In [ ]:
# Unzip the file
run(`unzip -o mclaughlin.prec.zip`)

Archive:  mclaughlin.prec.zip
  inflating: mclaughlin.prec         


Process(`unzip -o mclaughlin.prec.zip`, ProcessExited(0))

In [ ]:
# Load precedences
precedence_dict = Dict{Int, Vector{Int}}()
for line in eachline("mclaughlin.prec")
    parts = split(line)
    block_id = parse(Int, parts[1])
    n_prec = parse(Int, parts[2])
    precedence_dict[block_id] = [parse(Int, parts[i]) for i in 3:(2+n_prec)]
end

sorted_prec = sort(collect(precedence_dict))
n = length(sorted_prec)

println("Loaded: $n precedences\n")
println("Block ID | Num Prec | Precedent Blocks")
println("-"^50)

# First 10
for i in 1:min(10, n)
    block_id, prec_list = sorted_prec[i]
    println("$(lpad(block_id, 8)) | $(lpad(length(prec_list), 8)) | $(join(prec_list, ", "))")
end

if n > 20
    println("... ($(n-20) rows omitted) ...")
end

# Last 10 (if more than 10 total)
if n > 10
    for i in max(11, n-9):n
        block_id, prec_list = sorted_prec[i]
        println("$(lpad(block_id, 8)) | $(lpad(length(prec_list), 8)) | $(join(prec_list, ", "))")
    end
end

Loaded: 2140342 precedences

Block ID | Num Prec | Precedent Blocks
--------------------------------------------------
       0 |        0 | 
       1 |        0 | 
       2 |        0 | 
       3 |        0 | 
       4 |        0 | 
       5 |        0 | 
       6 |        0 | 
       7 |        0 | 
       8 |        0 | 
       9 |        0 | 
... (2140322 rows omitted) ...
 2140332 |       23 | 2098892, 2057453, 2057451, 2057312, 2016014, 2016010, 2057311, 2057313, 2015732, 1974575, 1974569, 2015870, 2015874, 2015731, 2015733, 1974152, 1933136, 1933128, 1974429, 1974435, 1974151, 1974153, 1932572
 2140333 |       23 | 2098893, 2057454, 2057452, 2057313, 2016015, 2016011, 2057312, 2057314, 2015733, 1974576, 1974570, 2015871, 2015875, 2015732, 2015734, 1974153, 1933137, 1933129, 1974430, 1974436, 1974152, 1974154, 1932573
 2140334 |       23 | 2098894, 2057455, 2057453, 2057314, 2016016, 2016012, 2057313, 2057315, 2015734, 1974577, 1974571, 2015872, 2015876, 2015733, 2015735, 1974154

### UPIT Optimization

In [ ]:
# Setup optimization
model = Model(Gurobi.Optimizer)
set_optimizer_attribute(model, "OutputFlag", 0)
set_optimizer_attribute(model, "MIPGap", 0.05)

idx = 1:nrow(blocks)
@variable(model, 0 <= x[idx] <= 1)
@objective(model, Max, sum(x[i] * blocks.blockvalue[i] for i in 1:nrow(blocks)))

# Add precedence constraints
id_to_index = Dict(blocks.id[i] => i for i in 1:nrow(blocks))
for i in 1:nrow(blocks)
    block_id = blocks.id[i]
    if haskey(precedence_dict, block_id)
        for prec_id in precedence_dict[block_id]
            if haskey(id_to_index, prec_id)
                @constraint(model, x[id_to_index[prec_id]] >= x[i])
            else
                @constraint(model, x[i] == 0)
                break
            end
        end
    end
end

# Solve and update upit column
start_time = time()
optimize!(model)

if termination_status(model) == MOI.OPTIMAL
    for i in 1:nrow(blocks)
        blocks.upit[i] = value(x[i]) > 0.5 ? 1 : 0
    end

    println("UPIT value: $(format(objective_value(model), commas=true, precision=0))")
    println("Extracted blocks: $(sum(blocks.upit))")
    println("Total Time: $(round((time() - start_time)/60, digits=2)) minutes")
else
    println("No optimal solution found")
end

CSV.write("upit_solution.csv", blocks)
println("Solution saved to: upit_solution.csv")

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2502611
Academic license 2502611 - for non-commercial use only - registered to dm___@demin.ufmg.br


LoadError: UndefVarError: `blocks` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

### 3D UPIT Plot

In [ ]:
# Create UPIT 3D plot
selected_blocks = blocks[blocks.upit .== 1, :]
unselected_blocks = blocks[blocks.upit .== 0, :]

# Selected blocks (upit = 1) - solid blue
trace_selected = scatter3d(
    x=selected_blocks.x,
    y=selected_blocks.y,
    z=selected_blocks.z,
    mode="markers",
    marker=attr(
        size=8,
        color="blue",
        opacity=1.0,
        symbol="square",
        line=attr(width=1, color="black")
    ),
    name="Selected (UPIT = 1)"
)

# Unselected blocks (upit = 0) - solid red
trace_unselected = scatter3d(
    x=unselected_blocks.x,
    y=unselected_blocks.y,
    z=unselected_blocks.z,
    mode="markers",
    marker=attr(
        size=8,
        color="red",
        opacity=1.0,
        symbol="square",
        line=attr(width=1, color="black")
    ),
    name="Unselected (UPIT = 0)"
)

# Create plot
layout = Layout(
    title="UPIT Results - 3D Block Plot",
    scene=attr(
        xaxis_title="X (m)",
        yaxis_title="Y (m)",
        zaxis_title="Z (m)",
        aspectmode="data"
    ),
    width=1000,
    height=600,
    margin=attr(l=0, r=150, b=80, t=100)
)

fig = Plot([trace_selected, trace_unselected], layout)
display(fig)
PlotlyJS.savefig(fig, "upit_results_3d.html")

### Sequential Algorithm for Open-Pit Mine Scheduling

In [ ]:
# Load UPIT solution
#blocks = CSV.read("upit_solution.csv", DataFrame)
#println("Loaded: $(nrow(blocks)) total blocks")

# Add period column to original blocks
blocks.period = Vector{Union{Int, Missing}}(missing, nrow(blocks))

# Filter only UPIT = 1 blocks for sequencing
#upit_blocks = blocks[blocks.upit .== 1, :]
#println("UPIT selected blocks: $(nrow(upit_blocks))")

# Load and filter precedences for UPIT blocks only
#precedence_dict = Dict{Int, Vector{Int}}()
#upit_ids = Set(upit_blocks.id)

#for line in eachline("mclaughlin.prec")
    #parts = split(line)
    #block_id = parse(Int, parts[1])

    #if block_id in upit_ids
        #n_prec = parse(Int, parts[2])
        #valid_precs = [parse(Int, parts[i]) for i in 3:(2+n_prec) if parse(Int, parts[i]) in upit_ids]
        #if !isempty(valid_precs)
            #precedence_dict[block_id] = valid_precs
        #end
    #end
#end
#println("Loaded: $(length(precedence_dict)) precedences for UPIT blocks")

# Parameters
MAX_PERIODS = 10
CAPACITY = 3_300_000
DISCOUNT_RATE = 0.15

# Initialize
extracted = Set{Int}()
period_results = []
start_time = time()

# Main optimization loop
for period in 0:(MAX_PERIODS-1)
    available_mask = [!(id in extracted) for id in blocks.id]
    available_indices = findall(available_mask)

    if isempty(available_indices)
        println("All UPIT blocks extracted by period $period")
        break
    end

    model = Model(Gurobi.Optimizer)
    set_optimizer_attribute(model, "OutputFlag", 0)
    set_optimizer_attribute(model, "MIPGap", 0.05)
    set_optimizer_attribute(model, "Threads", min(Sys.CPU_THREADS, 8))

    @variable(model, x[1:length(available_indices)], Bin)
    @objective(model, Max, sum(x[i] * blocks.blockvalue[available_indices[i]] for i in eachindex(available_indices)))

    # Add precedence constraints
    id_to_var = Dict(blocks.id[available_indices[i]] => i for i in eachindex(available_indices))
    for i in eachindex(available_indices)
        block_id = blocks.id[available_indices[i]]
        if haskey(precedence_dict, block_id)
            for prec_id in precedence_dict[block_id]
                if prec_id ∈ extracted
                    continue
                elseif haskey(id_to_var, prec_id)
                    @constraint(model, x[id_to_var[prec_id]] ≥ x[i])
                else
                    @constraint(model, x[i] == 0)
                    break
                end
            end
        end
    end

    # Capacity constraint
    @constraint(model, sum(blocks.destination[available_indices[i]] * blocks.tonn[available_indices[i]] * x[i]
               for i in eachindex(available_indices)) ≤ CAPACITY)

    # Solve
    opt_start = time()
    optimize!(model)
    opt_time = time() - opt_start

    if termination_status(model) == MOI.OPTIMAL
        undiscounted_value = objective_value(model)
        discounted_value = undiscounted_value / (1 + DISCOUNT_RATE)^period

        # Process selected blocks
        files_start = time()
        selected_blocks = Int[]
        for i in eachindex(available_indices)
            if value(x[i]) > 0.5
                idx = available_indices[i]
                block_id = blocks.id[idx]
                push!(selected_blocks, block_id)
                push!(extracted, block_id)
                blocks.period[idx] = period
            end
        end
        files_time = time() - files_start

        # Store results
        push!(period_results, (period, undiscounted_value, discounted_value, length(selected_blocks), opt_time, files_time))

        # Print formatted results
        println("Period: $period | " *
               "Undiscounted Value: $(format(undiscounted_value, commas=true, precision=0)) | " *
               "Discounted Value: $(format(discounted_value, commas=true, precision=0)) | " *
               "Opt Time (minutes): $(round(opt_time/60, digits=2)) | " *
               "Files Time (minutes): $(round(files_time/60, digits=2))")
    else
        println("No optimal solution found in period $period")
        break
    end
end

# Update original dataframe with periods
for row in eachrow(blocks)
    if !ismissing(row.period)
        blocks_idx = findfirst(blocks.id .== row.id)
        if blocks_idx !== nothing
            blocks.period[blocks_idx] = row.period  # Corrigido: usar blocks em vez de upit_blocks
        end
    end
end

# Calculate and print final results
npv = sum(r[3] for r in period_results)
total_time = (time() - start_time) / 60

println("\nNPV: $(format(npv, commas=true, precision=0))")
println("Total blocks extracted: $(length(extracted))")
println("Total time: $(round(total_time, digits=2)) minutes")

# Save solution
CSV.write("SeqCPIT_solution.csv", upit_blocks)
println("\nSolution saved to: SeqCPIT_solution.csv")

### Summary Results

In [ ]:
println("\n" * "="^100)
println("BLOCK SCHEDULING RESULTS")
println("="^100)

# Create DataFrame for summary table
summary_data = []
for (i, r) in enumerate(period_results)
    period, undiscounted, discounted, blocks, opt_time, files_time = r

    # Calculate period metrics
    period_mask = (upit_blocks.period .== period) .& (.!ismissing.(upit_blocks.period))
    period_blocks = upit_blocks[period_mask, :]

    mine_cap = sum(period_blocks.tonn)

    # Filter processed blocks
    proc_mask = period_mask .& (upit_blocks.destination .== 1)
    proc_blocks = upit_blocks[proc_mask, :]
    proc_cap = isempty(proc_blocks) ? 0.0 : sum(proc_blocks.tonn)

    # Calculate average Cu grade only for processed blocks
    cu_grade = isempty(proc_blocks) ? 0.0 : mean(proc_blocks.cu)

    push!(summary_data, (
        period,
        format(round(Int, discounted), commas=true, precision=0),          # Format with thousands separator
        format(round(Int, mine_cap), commas=true, precision=0), # Format with thousands separator
        format(round(Int, proc_cap), commas=true, precision=0), # Format with thousands separator
        format(cu_grade, precision=3),            # 3 decimal places
        format(opt_time/60, precision=2),            # 2 decimal place
        format(files_time/60, precision=2),          # 2 decimal place
        format(opt_time/60 + files_time/60, precision=2) # 2 decimal place
    ))
end

# Create DataFrame
summary_df = DataFrame(
    Period = [r[1] for r in summary_data],
    Value = [r[2] for r in summary_data],
    Mine_Cap_Mt = [r[3] for r in summary_data],
    Proc_Cap_Mt = [r[4] for r in summary_data],
    Cu_Grade = [r[5] for r in summary_data],
    Opt_Time = [r[6] for r in summary_data],
    Files_Time = [r[7] for r in summary_data],
    Total_Time = [r[8] for r in summary_data]
)

# Display formatted table
pretty_table(summary_df,
    header = ["Period", "Value", "Mine Cap", "Proc Cap", "Cu Grade",
              "Opt Time", "Files Time", "Total Time"],
    alignment = :c,
    crop = :none,
    formatters = ft_printf("%s")  # Preserve formatted strings
)

# Calculate totals
ore_mask = (upit_blocks.destination .== 1) .& (.!ismissing.(upit_blocks.period))
waste_mask = (upit_blocks.destination .== 0) .& (.!ismissing.(upit_blocks.period))

total_ore = sum(ore_mask)
total_waste = sum(waste_mask)
total_blocks = total_ore + total_waste
total_npv = sum(r[3] for r in period_results)

println("\n" * "="^100)
println("NPV: ", format(total_npv, commas=true, precision=0))
println("Ore blocks: ", format(total_ore, commas=true))
println("Waste blocks: ", format(total_waste, commas=true))
println("Total blocks: ", format(total_blocks, commas=true))
println("Total time: ", format(total_time, precision=2), " minutes")
println("="^100)

### Capacity & Grade Chart

In [ ]:
# Extract data from summary_data
periods = [r[1] for r in summary_data]
mine_cap = [parse(Float64, replace(r[3], "," => "")) / 1_000_000 for r in summary_data]  # Convert to millions of tons
proc_cap = [parse(Float64, replace(r[4], "," => "")) / 1_000_000 for r in summary_data]   # Convert to millions of tons
cu_grade = [parse(Float64, r[5]) for r in summary_data]

# Create traces for the bars
trace_mine = bar(
    x=periods,
    y=mine_cap,
    name="Mine Capacity (Mt)",
    marker_color="orange",
    opacity=0.7,
    width=0.35,
    offset=-0.175,
    marker_line_color="black",
    marker_line_width=1.5
)

trace_proc = bar(
    x=periods,
    y=proc_cap,
    name="Process Capacity (Mt)",
    marker_color="blue",
    opacity=0.7,
    width=0.35,
    offset=0.175,
    marker_line_color="black",
    marker_line_width=1.5
)

# Create trace for Cu grade (secondary axis)
trace_cu = scatter(
    x=periods,
    y=cu_grade,
    name="Cu Grade (%)",
    line_color="green",
    line_width=3,
    marker_size=8,
    marker_line_color="black",
    marker_line_width=1.5,
    yaxis="y2"
)

# Create the layout
layout = Layout(
    title="Mining Capacities and Cu Grade by Period",
    xaxis_title="Period",
    yaxis_title="Capacity (Millions of Tons)",
    yaxis2_title="Cu Grade (%)",
    yaxis2=attr(
        overlaying="y",
        side="right",
        showgrid=false
    ),
    barmode="group",
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis=attr(
        tickvals=periods,
        showline=true,
        linecolor="black",
        mirror=true
    ),
    yaxis=attr(
        showline=true,
        linecolor="black",
        mirror=true
    ),
    legend=attr(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    hovermode=false,
    dragmode=false
)

# Create the plot
p = Plot([trace_mine, trace_proc, trace_cu], layout)

# Salve image (PNG)
using PlotlyJS
savefig(p, "mining_capacity_plot.png", format="png", width=1000, height=600, scale=2)

# Display
display(p)

### SeqCPIT 3D Plot

In [ ]:
# Prepare data for 3D plot - using upit_blocks
blocks_with_period = upit_blocks[.!ismissing.(upit_blocks.period), :]
unique_periods = sort(unique(skipmissing(blocks_with_period.period)))

# Create automatic color scale using Plotly's built-in scales
color_scale = "Rainbow"

# Create traces for each period
traces = GenericTrace[]
for (i, p) in enumerate(unique_periods)
    period_blocks = blocks_with_period[blocks_with_period.period .== p, :]
    trace = scatter3d(
        x=period_blocks.x,
        y=period_blocks.y,
        z=period_blocks.z,
        mode="markers",
        marker=attr(
            size=8,
            color=p,  # Using period number to map to color scale
            colorscale=color_scale,
            symbol="square",
            opacity=1,
            line=attr(width=0.5, color="black"),
            colorbar=attr(title="Period"),
            showscale=false  # We'll show one colorbar for all traces
        ),
        name="Period $p"
    )
    push!(traces, trace)
end

# Create layout with improved viewing angle and right-side legend
layout = Layout(
    title=attr(
        text="Block Scheduling by Period",
        x=0.5,
        xanchor="center",
        font=attr(size=14)
    ),
    scene=attr(
        xaxis_title="X (m)",
        yaxis_title="Y (m)",
        zaxis_title="Elevation (m)",
        aspectmode="data",
        camera=attr(
            eye=attr(x=1.5, y=1.5, z=1.0)  # Adjust viewing angle
        )
    ),
    width=1000,
    height=600,
    margin=attr(l=0, r=150, b=80, t=100),  # Extra space on right for legend
    legend=attr(
        orientation="v",
        yanchor="middle",
        y=0.5,
        xanchor="left",
        x=1.05,  # Position legend outside to the right
        title_text="Periods",
        font=attr(size=10)
    )
)

# Create and display the plot
fig = Plot(traces, layout)
display(fig)

# Save interactive plot
savefig(fig, "SeqCPIT_3D_Plot.html")